In [4]:
import pandas as pd
import re
import unicodedata
import csv
from newspaper import Article
from tqdm import tqdm

# 1. 판다스에 tqdm 기능을 연결합니다.
tqdm.pandas()

def clean_text_robust(text):
    if pd.isna(text):
        return ""
    # 유니코드 정규화 (NFC)
    text = unicodedata.normalize('NFC', str(text))
    # 제어 문자 및 줄바꿈/탭 제거
    text = re.sub(r'[\x00-\x1f\x7f-\x9f\n\r\t]', ' ', text)
    # 특수 기호 제거 (필요한 문자만 유지)
    text = re.sub(r'[^a-zA-Z0-9가-힣\s.,!?\'\"\-]', '', text)
    # 공백 정리
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def safe_fetch_and_clean(row):
    target_text = str(row['text']) if pd.notna(row['text']) else ""
    
    if target_text.strip() == "":
        try:
            article = Article(row['url'], language='ko')
            article.download()
            article.parse()
            target_text = article.text
        except Exception:
            return ""
            
    return clean_text_robust(target_text)

# --- 파일 불러오기 및 적용 ---
input_file = r"C:\Users\com\Downloads\더불어민주당보도자료_2003-11-11_2026-02-04.csv"  # 파일명을 입력하세요
output_file = r"C:\Users\com\Downloads\더불어민주당보도자료_2003-11-11_2026-02-04_cleaned.csv"  # 저장할 파일명을 입력하세요

# 데이터 불러오기
df = pd.read_csv(input_file, encoding='utf-8', engine='python', on_bad_lines='warn')

if 'text' in df.columns and 'url' in df.columns:
    print("⏳ 데이터를 처리 중입니다. 잠시만 기다려 주세요...")
    
    # 2. apply 대신 progress_apply를 사용하면 진행바가 나타납니다!
    df['text'] = df.progress_apply(safe_fetch_and_clean, axis=1)
    
    # 결과 저장 (데이터 분할 방지 옵션 포함)
    df.to_csv(
        output_file, 
        index=False, 
        encoding='utf-8-sig', 
        quoting=csv.QUOTE_ALL, 
        escapechar='\\'
    )
    print(f"\n✅ 작업 완료! 파일이 저장되었습니다: {output_file}")
else:
    print("❌ 컬럼명을 확인해 주세요 ('text', 'url')")

⏳ 데이터를 처리 중입니다. 잠시만 기다려 주세요...


100%|██████████| 55497/55497 [06:17<00:00, 147.03it/s]  



✅ 작업 완료! 파일이 저장되었습니다: C:\Users\com\Downloads\더불어민주당보도자료_2003-11-11_2026-02-04_cleaned.csv
